In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, mean, round

spark = SparkSession.builder.appName("StudentPerformanceAnalysis").getOrCreate()

# Load the dataset using your DATA folder path
filepath = "D:/ABD0008/DATA/students.csv"   
df = spark.read.csv(filepath, header=True, inferSchema=True)
df.printSchema()

root
 |-- gender: string (nullable = true)
 |-- race/ethnicity: string (nullable = true)
 |-- parental level of education: string (nullable = true)
 |-- lunch: string (nullable = true)
 |-- test preparation course: string (nullable = true)
 |-- math score: integer (nullable = true)
 |-- reading score: integer (nullable = true)
 |-- writing score: integer (nullable = true)



In [13]:
df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|female|  518|
|  male|  482|
+------+-----+



In [14]:
df.select("race/ethnicity").distinct().show()

+--------------+
|race/ethnicity|
+--------------+
|       group B|
|       group C|
|       group D|
|       group A|
|       group E|
+--------------+



In [15]:
df.select("parental level of education").distinct().show(truncate=False)

+---------------------------+
|parental level of education|
+---------------------------+
|some high school           |
|associate's degree         |
|high school                |
|bachelor's degree          |
|master's degree            |
|some college               |
+---------------------------+



In [16]:
q4_df = df.filter(
    (col("gender") == "female") &
    (col("math score") > 79) &
    (col("parental level of education") == "high school")
)

print(f"Number of students: {q4_df.count()}")
q4_df.select("gender", "parental level of education", "math score").show()

Number of students: 5
+------+---------------------------+----------+
|gender|parental level of education|math score|
+------+---------------------------+----------+
|female|                high school|        87|
|female|                high school|        99|
|female|                high school|        88|
|female|                high school|        81|
|female|                high school|        81|
+------+---------------------------+----------+



In [17]:
df.groupBy("gender").agg(
    round(mean("math score"), 2).alias("avg_math_score")
).show()

+------+--------------+
|gender|avg_math_score|
+------+--------------+
|female|         63.63|
|  male|         68.73|
+------+--------------+



In [18]:
df.groupBy("gender").agg(
    round(mean("reading score"), 2).alias("avg_reading_score")
).show()

+------+-----------------+
|gender|avg_reading_score|
+------+-----------------+
|female|            72.61|
|  male|            65.47|
+------+-----------------+



In [19]:
overall_df = df.withColumn(
    "overall_score", 
    (col("math score") + col("reading score") + col("writing score")) / 3
)

overall_df.groupBy("parental level of education").agg(
    round(mean("math score"), 2).alias("avg_math"),
    round(mean("reading score"), 2).alias("avg_reading"),
    round(mean("writing score"), 2).alias("avg_writing"),
    round(mean("overall_score"), 2).alias("avg_overall")
).orderBy(col("avg_overall").desc()).show(truncate=False)

+---------------------------+--------+-----------+-----------+-----------+
|parental level of education|avg_math|avg_reading|avg_writing|avg_overall|
+---------------------------+--------+-----------+-----------+-----------+
|master's degree            |69.75   |75.37      |75.68      |73.6       |
|bachelor's degree          |69.39   |73.0       |73.38      |71.92      |
|associate's degree         |67.88   |70.93      |69.9       |69.57      |
|some college               |67.13   |69.46      |68.84      |68.48      |
|some high school           |63.5    |66.94      |64.89      |65.11      |
|high school                |62.14   |64.7       |62.45      |63.1       |
+---------------------------+--------+-----------+-----------+-----------+



In [20]:
no_prep_high_math = df.filter(
    (col("test preparation course") == "none") &
    (col("math score") > 70)
)

print(f"Total matching records: {no_prep_high_math.count()}")
no_prep_high_math.show(20, truncate=False)

Total matching records: 223
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|lunch       |test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|female|group B       |bachelor's degree          |standard    |none                   |72        |72           |74           |
|female|group B       |master's degree            |standard    |none                   |90        |95           |93           |
|male  |group C       |some college               |standard    |none                   |76        |78           |75           |
|female|group B       |associate's degree         |standard    |none                   |71        |83           |78           |
|male  |group C       |high school                |standard    |none        